# BGE Reranker v2-m3

## 一、项目简介

### 1.1 官方项目简介

BGE Reranker v2-m3 是 BAAI（北京智源人工智能研究院）推出的多语言 Cross-Encoder 重排序模型。基于 XLM-RoBERTa 架构，支持 100+ 种语言，适用于 RAG 检索增强生成流程中的精排阶段：先 Embedding 粗召回（如搭配 [gte-Qwen2-1.5B](https://huggingface.co/Alibaba-NLP/gte-Qwen2-1.5B-instruct)），再用本 Reranker 精排。

| 属性 | 值 |
|------|-----|
| 参数量 | ~560M |
| 架构 | XLM-RoBERTa (Cross-Encoder) |
| 上下文长度 | 8192 tokens |
| 支持语言 | 100+ 种 |
| 显存需求 | ~4GB（float32） |
| 模型体积 | ~2.2GB |
| 输入格式 | (query, document) 对 |
| 输出 | 相关性分数（原始 logit，越大越相关） |

### 1.2 本项目简介

本项目在趋动云平台部署 BGE Reranker v2-m3，提供 Gradio WebUI 重排序界面。用户输入查询语句和候选文档列表（一行一条），模型按相关性从高到低排序返回。

| 属性 | 值 |
|------|-----|
| 推理框架 | PyTorch 2.1.1 + Transformers 4.35.2 |
| Web UI | Gradio 4.7.1 |
| Python | 3.10 |
| CUDA | 12.1 |
| 实现类型 | 推理 + Gradio WebUI |
| 输出 | float32，单 logit 相关性分数 |

## 二、官方链接

- **HuggingFace**: https://huggingface.co/BAAI/bge-reranker-v2-m3
- **BAAI Embedding 系列**: https://huggingface.co/BAAI
- **FlagEmbedding 工具库**: https://github.com/FlagOpen/FlagEmbedding

## 三、算力推荐

| 场景 | 推荐规格 | 显存 |
|------|------|:--:|
| 最低运行 | T4 | 4GB |
| 流畅体验 | A10 | 8GB+ |
| 批量处理 | A100 40GB | 40GB |

> 已验证：T4 16GB 规格，加载模型约 5 秒，单次重排 5 条文档约 0.5 秒。

## 四、使用说明

必须先开放 **7860** 端口，一键启动：

```bash
cd /gemini/code && bash start.sh
```

### Web 界面

- 输入 Query（查询语句）和 Documents（候选文档，一行一条）
- 可选设置 Top-K 只显示前 K 条结果
- 点击 **Rerank**，首次触发模型加载（约 5 秒），后续秒出

### 效果展示

Query: "What is deep learning?" — 模型准确将深度学习相关文档排在前列：

![效果截图](SP.png)

### Jupyter 内手动调用

如需在 Jupyter 中直接调用模型进行重排序：

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = "1"

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "/gemini/pretrain/bge-reranker-v2-m3"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    local_files_only=True,
)
model = model.cuda()
model.eval()
print(f"Model loaded on: {next(model.parameters()).device}")

In [ ]:
def rerank(query, documents, top_k=None):
    pairs = [[query, doc] for doc in documents]

    with torch.no_grad():
        inputs = tokenizer(
            pairs,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(model.device)

        scores = model(**inputs, return_dict=True).logits.view(-1).float()
        scores = scores.cpu().tolist()

    results = list(zip(documents, scores))
    results.sort(key=lambda x: x[1], reverse=True)

    if top_k:
        results = results[:top_k]

    return results


query = "What is deep learning?"
candidates = [
    "Deep learning is a subset of machine learning that uses neural networks with many layers.",
    "Machine learning is a field of artificial intelligence.",
    "Neural networks are inspired by the human brain structure.",
    "Python is a popular programming language.",
    "The weather today is sunny and warm.",
]

results = rerank(query, candidates)
for rank, (doc, score) in enumerate(results, 1):
    print(f"[Rank {rank}] Score: {score:.4f} | {doc}")

预期输出：

```
[Rank 1] Score: -3.6995 | Deep learning is a subset of machine learning...
[Rank 2] Score: -4.9584 | Machine learning is a field of artificial intelligence.
[Rank 3] Score: -6.7616 | Neural networks are inspired by the human brain structure.
[Rank 4] Score: -10.5182 | Python is a popular programming language.
[Rank 5] Score: -11.0378 | The weather today is sunny and warm.
```

> 分数为原始 logit（未归一化），数值越大表示越相关。如需概率，对分数做 `sigmoid` 即可。